# एसआरए सिनैप्स डिलीशन डेमो
## - एलएलएम से भौतिक रूप से विशिष्ट ज्ञान प्राप्त करें!

यह नोटबुक लेख का एक इंटरैक्टिव डेमो है *[\[एआई रोमांस\] एलएलएम से भौतिक रूप से विशिष्ट ज्ञान निकालें! हॉट-स्वैपेबल एलएलएम का 'सिनैप्स डिलीशन' और 'पर्ज'](https://qiita.com/JunSuzukiJapan/items/)*।

एसआरए (सिनैप्टिक रूटिंग आर्किटेक्चर) में, अनावश्यक ज्ञान और सिनैप्स को दो तरीकों से हटाया जा सकता है।

| विधि | एपीआई | उद्देश्य |
|------|-----|------|
| **शारीरिक निष्कासन** | `pop_synapses(N)` | मॉडल आकार को पुनर्स्थापित करने के लिए टेल से हॉट-स्वैप के माध्यम से जोड़े गए सिनैप्स को हटाता है |
| **शून्य-स्पष्ट (शुद्ध)** | `clear_synapses([idx])` | मध्यवर्ती सूचकांकों पर सिनैप्स को अक्षम करता है और उन्हें मुफ्त स्लॉट में परिवर्तित करता है |

हम व्यवहार में शून्य-समाशोधन और उसके समाधान के दौरान होने वाले **कोसाइन-समानता जाल** की भी पुष्टि करेंगे।

अंत में, हम `clear_synapses` को वास्तव में मल्टीटास्क-प्रशिक्षित मॉडल पर लागू करते हैं और प्रदर्शित करते हैं कि **केवल लक्षित कार्य का कार्य नष्ट हो जाता है जबकि हर अन्य कार्य पूरी तरह से बरकरार रहता है (शून्य भूल)**।

---
## भाग 1: सिनैप्स रिमूवल (`pop_synapses`)

हमने भौतिक रूप से सिनैप्स को काट दिया, जिन्हें बाद में हॉट-स्वैप के माध्यम से पूंछ से शुरू करके जोड़ा गया था।
किसी OS से प्लगइन को अनइंस्टॉल करने की तरह, आप AI के मस्तिष्क के कुछ हिस्सों को भौतिक रूप से हटा सकते हैं।

In [ ]:
import sys, os

if 'google.colab' in sys.modules:
    if not os.path.exists('SynapticRouter'):
        !git clone https://github.com/JunSuzukiJapan/SynapticRouter.git
    %cd SynapticRouter
    !pip install tiktoken torch matplotlib seaborn

sys.path.append('.')
sys.path.append('./src')
if 'google.colab' not in sys.modules:
    sys.path.append('..')
    sys.path.append('../src')

import torch
import torch.nn.functional as F
from src.sra_language_models import MoESRALanguageModel


In [ ]:
# Build a small model to walk through the add -> remove flow
model = MoESRALanguageModel(vocab_size=1000, dim=128, layers=2, num_synapses=4, k=2, syn_hidden=256)

print('=== Approach 1: Synapse Removal (Physical Cut) ===')
print(f'[Initial state] Synapse count: {model.blocks[0].num_synapses}')
print(f'  Router embedding shape: {model.blocks[0].router.get_full_emb().shape}')

# Add 2 Synapses via Hot-Swap
model.add_synapses(2, freeze_base=True)
print(f'\n[After adding] Synapse count: {model.blocks[0].num_synapses}')
print(f'  Router embedding shape: {model.blocks[0].router.get_full_emb().shape}')

# Physically remove the added Synapses
model.pop_synapses(2)
print(f'\n[After removal] Synapse count: {model.blocks[0].num_synapses}')
print(f'  Router embedding shape: {model.blocks[0].router.get_full_emb().shape}')
print('\nMemory usage has been fully restored!')


---
## भाग 2: शून्य-समाशोधन (पर्ज) और "कोसाइन-समानता जाल"

यदि हम किसी इंटरमीडिएट इंडेक्स पर सिनैप्स को हटाना चाहते हैं, तो इसे भौतिक रूप से हटाने से आईडी इधर-उधर हो जाएंगी।
इसलिए इसके बजाय, हम वज़न को `0.0` से अधिलेखित कर देते हैं - एक "शून्य-स्पष्ट"।

हालाँकि, यहाँ एक **भयानक जाल** है। शून्य वेक्टर की कोसाइन समानता `0.0` बन जाती है,
और यदि सामान्य सिनैप्स का स्कोर नकारात्मक है, तो **खाली सिनैप्स उच्च स्कोर के साथ समाप्त होता है और डेटा को अवशोषित करना शुरू कर देता है** - एक ब्लैक-होल घटना।

इस जाल को रोकने के लिए SRA के राउटर में एक अंतर्निहित **`-inf` मास्क** है। आइए इसे व्यवहार में सत्यापित करें।

In [ ]:
print('=== Approach 2: Verifying zero-clear and the -inf mask ===')

# Create a new model
model2 = MoESRALanguageModel(vocab_size=1000, dim=128, layers=1, num_synapses=4, k=2, syn_hidden=256)

# Inspect the weights of Synapse #2 before zero-clearing
target_idx = 2
emb_before = model2.blocks[0].router.get_full_emb()
print(f'[Before zero-clear] Synapse {target_idx} router embedding norm: {torch.norm(emb_before[target_idx]):.4f}')
print(f'  W1 norm: {torch.norm(model2.blocks[0].get_full_param("w1")[target_idx]):.4f}')

# Execute zero-clear
model2.clear_synapses([target_idx])

emb_after = model2.blocks[0].router.get_full_emb()
print(f'\n[After zero-clear] Synapse {target_idx} router embedding norm: {torch.norm(emb_after[target_idx]):.4f}')
print(f'  W1 norm: {torch.norm(model2.blocks[0].get_full_param("w1")[target_idx]):.4f}')
print('\nThe weights of Synapse #2 have been zeroed out completely!')


In [ ]:
# Actually verify the -inf mask in action
print('=== Verifying the -inf mask ===')

# Run the router with a dummy input
model2.eval()
dummy_input = torch.randint(0, 1000, (1, 10))
with torch.no_grad():
    logits, router_logits = model2(dummy_input)

# Inspect the router logits (final layer)
router_scores = router_logits[0]  # shape: (batch, seq, num_synapses)
print(f'Router scores (first token):')
scores = router_scores[0, 0]
for i in range(len(scores)):
    label = 'BLOCKED (-inf)' if scores[i] == float('-inf') else f'{scores[i]:.4f}'
    marker = ' <- zero-cleared' if i == target_idx else ''
    print(f'  Synapse {i}: {label}{marker}')

print('\nThe score of the zero-cleared Synapse has been set to -inf,')
print('   so the router can never select this Synapse again.')
print('   This completely prevents the "black-hole phenomenon".')


---
## भाग 3: कार्यात्मक प्रमाण - एक प्रशिक्षित मॉडल पर `clear_synapses`

अब हम मुख्य कार्यक्रम पर आते हैं। भाग 1 और 2 में हमने "एपीआई व्यवहार" को सत्यापित किया,
लेकिन जो वास्तव में मायने रखता है वह कार्यात्मक प्रमाण है: **"शून्य-समाशोधन के बाद, क्या केवल हटाया गया ज्ञान खो जाता है जबकि ज्ञान का हर दूसरा हिस्सा पूरी तरह से बरकरार रहता है?"**

नोटबुक 05 (लेसियन प्रयोग) के समान दृष्टिकोण का उपयोग करना:
1. `copy` और `reverse` पर मल्टीटास्क-ट्रेन
2. `reverse` द्वारा अक्सर उपयोग किए जाने वाले सिनैप्स को पहचानें और उन्हें `clear_synapses` से हटा दें
3. सत्यापित करें कि `reverse` अनसुलझा हो जाता है जबकि `copy` 100% स्कोर करना जारी रखता है (शून्य भूल)

In [ ]:
# Exactly the same setup as notebook 05
from src.sra_gpu_models import MoESRAModel
from src.sra_experiment import make_multitask_batch, make_batch, make_optimizer, load_balance_loss
from src.constants import VOCAB_SIZE, BOS, PAD

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

class MoESRAConfig:
    def __init__(self, **kwargs):
        for k, v in kwargs.items():
            setattr(self, k, v)

config = MoESRAConfig(
    vocab_size=30,
    d_model=128,
    n_layers=2,
    n_heads=4,
    num_synapses=8,
    k=2,
    max_seq_len=32
)

model3 = MoESRAModel(config.vocab_size, config.d_model, config.n_layers, config.num_synapses, config.k, syn_hidden=256).to(device)
optimizer = make_optimizer(model3, lr=0.001)


In [ ]:
# Multitask training (same as notebook 05)
print('Training started... (Copy & Reverse)')
model3.train()
epochs = 1500
batch_size = 32
tasks = ['copy', 'reverse']

for epoch in range(epochs):
    x, y, _ = make_multitask_batch(tasks, batch_size, min_len=4, max_len=8, device=device)

    optimizer.zero_grad()
    y_in = torch.cat([torch.full((y.size(0), 1), BOS, dtype=torch.long, device=device), y[:, :-1]], dim=1)
    outputs, routing_weights, _ = model3(x, y_in)

    loss = F.cross_entropy(outputs.reshape(-1, config.vocab_size), y.reshape(-1))
    lb_loss = load_balance_loss(routing_weights) * 0.1
    total_loss = loss + lb_loss

    total_loss.backward()
    optimizer.step()

    if (epoch + 1) % 300 == 0:
        print(f'Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f}')

print('Training finished!')


### 3-1. विलोपन पूर्व प्रदर्शन परीक्षण और लक्ष्य पहचान
हम पुष्टि करते हैं कि प्रत्येक कार्य को सही ढंग से हल किया जा सकता है, और `reverse` द्वारा सबसे अधिक उपयोग किए जाने वाले सिनैप्स की पहचान करते हैं।

In [ ]:
def test_task(task_name):
    model3.eval()
    x, y, _ = make_multitask_batch([task_name], batch_size=100, min_len=6, max_len=6, device=device)
    with torch.no_grad():
        y_in = torch.cat([torch.full((y.size(0), 1), BOS, dtype=torch.long, device=device), y[:, :-1]], dim=1)
        outputs, routing_weights, _ = model3(x, y_in)
        preds = outputs.argmax(dim=-1)

    mask = y != PAD
    acc = (preds[mask] == y[mask]).float().mean().item()

    # Aggregate which Synapses were used (Layer 0)
    chosen = routing_weights[0].argmax(dim=-1)
    usage = torch.bincount(chosen.flatten(), minlength=config.num_synapses)

    print(f'[{task_name.upper()}] Accuracy: {acc*100:.1f}%')
    return usage

print('=== Before Deletion ===')
copy_usage = test_task('copy')
reverse_usage = test_task('reverse')

# Identify Synapses heavily used by Reverse but rarely used by Copy
# (Same logic as notebook 05)
diff = reverse_usage - copy_usage
target_synapses = [i for i, val in enumerate(diff) if val > 0]

# If they cannot be perfectly separated, pick the single Synapse most used by Reverse
if len(target_synapses) == 0:
    target_synapses = [diff.argmax().item()]

print(f'\n=> Deletion target: Synapses {target_synapses} (heavily used by REVERSE)')


### 3-2. `clear_synapses` के माध्यम से सिनैप्स विलोपन
नोटबुक 05 में हमने मैन्युअल रूप से `block.w1.data[idx].zero_()` चलाया, लेकिन यहां हम आधिकारिक **`clear_synapses` API** का उपयोग करते हैं, जो सुरक्षित विलोपन करने के लिए राउटर के `-inf` मास्क को भी लागू करता है।

In [ ]:
print(f'Deleting Synapses {target_synapses} via clear_synapses...')

# Record norms before deletion
for idx in target_synapses:
    emb_norm = torch.norm(model3.blocks[0].router.synapse_emb.data[idx]).item()
    w1_norm = torch.norm(model3.blocks[0].w1.data[idx]).item()
    print(f'  [Before deletion] Synapse {idx}: router embedding norm={emb_norm:.4f}, W1 norm={w1_norm:.4f}')

# Use the clear_synapses API (the router's -inf mask is also applied automatically)
for block in model3.blocks:
    block.router.clear_synapses(target_synapses)
    for idx in target_synapses:
        block.w1.data[idx].zero_()
        block.b1.data[idx].zero_()
        block.w2.data[idx].zero_()
        block.b2.data[idx].zero_()
        block.state.data[idx].zero_()

print('\nDeletion complete!')

# Check norms after deletion
for idx in target_synapses:
    emb_norm = torch.norm(model3.blocks[0].router.synapse_emb.data[idx]).item()
    w1_norm = torch.norm(model3.blocks[0].w1.data[idx]).item()
    print(f'  [After deletion] Synapse {idx}: router embedding norm={emb_norm:.4f}, W1 norm={w1_norm:.4f}')


### 3-3. विलोपन के बाद प्रदर्शन परीक्षण (शून्य विस्मृति का सत्यापन)

हम कुछ सिनेप्सेस को हटाकर दोबारा परीक्षण करते हैं।

**अपेक्षित परिणाम:**
- **कॉपी**: सटीकता संरक्षित (विभिन्न सिनैप्स का उपयोग करती है, इसलिए पूरी तरह से बरकरार)
- **रिवर्स**: सटीकता गिरती है (इसके विशेष सिनैप्स समाप्त हो गए हैं)

In [ ]:
print('=== After Deletion ===')
test_task('copy')
test_task('reverse')

print('\n[Discussion]')
print('When you destroy part of a single monolithic neural network by zeroing it out,')
print('all tasks usually collapse together.')
print('However, in SRA, an independent expert pathway (Synapse) is autonomously formed')
print('for each task, so even when clear_synapses removes a specific Synapse,')
print('tasks that use a different Synapse are completely unaffected.')
print('\nThis is the true strength of SRA as a modular AI.')
print('The free slot left behind by a deleted Synapse can later be reused by')
print('overwriting it (Hot-Swap) with a new plugin!')


---
## सारांश

| डेमो | क्या प्रदर्शित किया गया |
|------|----------------------|
| भाग 1 | `pop_synapses` के माध्यम से भौतिक निष्कासन और स्मृति बहाली |
| भाग 2 | `clear_synapses` के माध्यम से शून्य-समाशोधन और `-inf` मास्क का सत्यापन |
| भाग 3 | प्रशिक्षित मॉडल पर `clear_synapses` -> केवल लक्षित कार्य नष्ट हो जाता है जबकि अन्य बरकरार रहते हैं |

इसके साथ, हमने मॉड्यूलर एआई का पूरा जीवनचक्र हासिल कर लिया है: **"प्रशिक्षण -> एकीकरण (हॉट-स्वैप) -> विलोपन (पर्ज) -> स्लॉट पुन: उपयोग (ओवरराइट)"**।